# Sophia AGI — LoRA training (Google Colab)

## Open this notebook from GitHub (not a stale copy)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomyimkc/sophia-agi/blob/main/notebooks/Sophia-LoRA-Colab.ipynb)

## Enable GPU **before** running code

1. **Runtime** → **Change runtime type** → **T4 GPU** → **Save**
2. **Runtime** → **Restart session**
3. Run every cell **in order** — do not skip the train cell

Train **Sophia** adapter on the provenance corpus with QLoRA (4-bit).

**Next:** [Sophia-LoRA-Eval-Colab.ipynb](https://colab.research.google.com/github/tomyimkc/sophia-agi/blob/main/notebooks/Sophia-LoRA-Eval-Colab.ipynb) · HF model: [tomyimkc/sophia-agi-lora-v1](https://huggingface.co/tomyimkc/sophia-agi-lora-v1)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Runtime → Change runtime type → T4 GPU → Save → Restart session"
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import os
import shutil
import subprocess

REPO = "sophia-agi"
REMOTE = "https://github.com/tomyimkc/sophia-agi.git"

if os.path.isdir(REPO):
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--depth", "1", REMOTE], check=True)
%cd sophia-agi

head = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True, check=True)
print("Repo commit:", head.stdout.strip())

In [ ]:
!pip install -q -U \
  "transformers>=4.44,<5.0" \
  "peft>=0.10,<0.18" \
  "datasets>=2.18" \
  "accelerate>=0.28" \
  "bitsandbytes>=0.43"

import transformers, peft, bitsandbytes
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL = "Qwen/Qwen2.5-7B-Instruct" if vram_gb > 20 else "Qwen/Qwen2.5-3B-Instruct"
BATCH_SIZE = 1
MAX_SEQ_LEN = 1024
EPOCHS = 3
print(f"model={MODEL} vram={vram_gb:.1f}GB batch={BATCH_SIZE} seq={MAX_SEQ_LEN} epochs={EPOCHS}")

In [ ]:
import subprocess
import sys
from pathlib import Path

CKPT = Path("training/lora/checkpoints/sophia-v1")
REQUIRED = ["adapter_config.json", "adapter_model.safetensors", ".train_complete"]

if CKPT.exists():
    import shutil
    shutil.rmtree(CKPT)

prep = subprocess.run([sys.executable, "tools/prepare_lora_dataset.py"], check=False)
if prep.returncode != 0:
    raise RuntimeError(f"prepare_lora_dataset failed (exit {prep.returncode})")

cmd = [
    sys.executable,
    "tools/train_lora.py",
    "--4bit",
    "--epochs", str(EPOCHS),
    "--model", MODEL,
    "--batch-size", str(BATCH_SIZE),
    "--max-seq-len", str(MAX_SEQ_LEN),
]
print("Running:", " ".join(cmd))
train = subprocess.run(cmd, check=False)
if train.returncode != 0:
    raise RuntimeError(f"train_lora failed (exit {train.returncode}). Scroll up for traceback.")

missing = [f for f in REQUIRED if not (CKPT / f).exists()]
if missing:
    raise RuntimeError(f"Training incomplete — missing {missing}")

print("Training complete:", sorted(p.name for p in CKPT.iterdir()))

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

CKPT = Path("training/lora/checkpoints/sophia-v1")
REQUIRED = ["adapter_config.json", "adapter_model.safetensors", ".train_complete"]
missing = [f for f in REQUIRED if not (CKPT / f).exists()]
if missing:
    raise RuntimeError(
        f"Cannot download — missing {missing}. Run the TRAIN cell above first."
    )

zip_path = Path("sophia-lora-v1.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive("sophia-lora-v1", "zip", str(CKPT))
zip_mb = round(zip_path.stat().st_size / 1e6, 2)
print("Zip size MB:", zip_mb)
if zip_mb < 1:
    raise RuntimeError("Zip too small — adapter weights were not saved.")
files.download("sophia-lora-v1.zip")

## After download (local)

```bash
unzip sophia-lora-v1.zip -d training/lora/checkpoints/sophia-v1
python tools/eval_local_model.py --adapter training/lora/checkpoints/sophia-v1 --with-gate
```